# 신문/문어체 코퍼스 로드


In [ ]:

# === corpus 구성 ===
from pathlib import Path
import io
import json
import random
import re
import zipfile

DATASET_NAMES = [
    "NIKL_NEWSPAPER_2024_v1.0.zip",
    "NIKL_NEWSPAPER_2023_v1.0.zip",
    "NIKL_NEWSPAPER_2022_v1.0.zip",
    "NIKL_WRITING_2024_v1.0.zip",
    "NIKL_WRITING_2023_v1.0.zip",
    "NIKL_WRITING_2022_v1.0.zip",
    "NIKL_NEWS_WRITING_v1.0.zip",
]
SEARCH_ROOTS = [Path('.'), Path('/content')]
DATASET_PATTERNS = ("*NEWSPAPER*.zip", "*WRITING*.zip", "*ARTICLE*.zip", "*ESSAY*.zip")

MAX_LEN = 512
MIN_PASSAGE_CHARS = 40
MAX_PASSAGE_CHARS = 1600
SHUFFLE = True
SEED = 42

def clean_text(text: str) -> str:
    text = text.replace("\u200b", "").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def is_korean(text: str) -> bool:
    kor = len(re.findall(r"[가-힣]", text))
    return kor >= max(6, int(0.3 * len(text)))

def iter_candidate_zips():
    seen = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in DATASET_NAMES:
            path = (root / name).expanduser()
            if path.exists():
                resolved = path.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    yield resolved
        for pattern in DATASET_PATTERNS:
            for path in root.glob(pattern):
                resolved = path.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    yield resolved

ZIP_PATHS = list(iter_candidate_zips())

passages = []
seen_passages = set()
rng = random.Random(SEED)

def register_passage(text: str) -> None:
    if text in seen_passages:
        return
    seen_passages.add(text)
    passages.append(text)

def extract_passages(obj):
    results = []

    def walk(node):
        if isinstance(node, dict):
            for value in node.values():
                walk(value)
        elif isinstance(node, list):
            for item in node:
                walk(item)
        elif isinstance(node, str):
            candidate = clean_text(node)
            if (
                candidate
                and MIN_PASSAGE_CHARS <= len(candidate) <= MAX_PASSAGE_CHARS
                and is_korean(candidate)
            ):
                results.append(candidate)

    walk(obj)
    return results

files_scanned = 0
for zip_path in ZIP_PATHS:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            for name in zf.namelist():
                lower = name.lower()
                if not lower.endswith((".json", ".jsonl", ".txt")):
                    continue
                files_scanned += 1
                try:
                    with zf.open(name) as f:
                        if lower.endswith('.txt'):
                            text = io.TextIOWrapper(f, encoding='utf-8').read()
                            for chunk in text.split("\n\n"):
                                candidate = clean_text(chunk)
                                if (
                                    candidate
                                    and MIN_PASSAGE_CHARS <= len(candidate) <= MAX_PASSAGE_CHARS
                                    and is_korean(candidate)
                                ):
                                    register_passage(candidate)
                        elif lower.endswith('.jsonl'):
                            for line in io.TextIOWrapper(f, encoding='utf-8'):
                                line = line.strip()
                                if not line:
                                    continue
                                data = json.loads(line)
                                for candidate in extract_passages(data):
                                    register_passage(candidate)
                        else:
                            data = json.load(io.TextIOWrapper(f, encoding='utf-8'))
                            for candidate in extract_passages(data):
                                register_passage(candidate)
                except Exception as exc:
                    print(f"[warn] failed to parse {zip_path.name}:{name}: {exc}")
    except FileNotFoundError:
        continue
    except zipfile.BadZipFile as exc:
        print(f"[warn] {zip_path} is not a valid zip file: {exc}")

if not passages:
    fallback_passages = [
        clean_text(
            "한국 언론은 최근 지역 소멸 문제를 다루며 지방자치단체의 대응과 정부의 지원 정책을 상세하게 분석했다."
        ),
        clean_text(
            "기후 변화에 대한 시민의 인식이 높아지면서 기업들은 탄소 배출을 줄이는 친환경 경영 전략을 앞다투어 도입하고 있다."
        ),
        clean_text(
            "한 대학의 글쓰기 강좌에서는 고전을 현대적으로 해석하는 에세이가 우수작으로 선정되며 학생들의 창의적 문장이 주목을 받았다."
        ),
    ]
    passages = fallback_passages
    print("No newspaper/writing corpus found; using fallback passages.")

if SHUFFLE and len(passages) > 1:
    rng.shuffle(passages)

print(f"ZIP files scanned: {len(ZIP_PATHS)} (files parsed: {files_scanned})")
print(f"Collected passages: {len(passages)}")
sample_preview = "<empty>"
if passages:
    snippet = passages[0][:120]
    sample_preview = snippet + ("..." if len(passages[0]) > 120 else "")
print("Sample:", sample_preview)

corpus_texts = passages


JSON files read: 3227
Total pairs: 43870
Sample: ('아니 근데 나 국수 먹고 싶다.', '오늘?')


# 토크나이저

In [ ]:

# sentencepiece tokenization

import sentencepiece as spm

if not corpus_texts:
    raise RuntimeError("corpus_texts is empty; ensure dataset is available.")

with open("corpus.txt", "w", encoding="utf-8") as f:
    for text in corpus_texts:
        f.write(text.replace("\n", " ") + "\n")

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="spm16k",
    vocab_size=16000,
    model_type="unigram",
    character_coverage=0.9995,
    pad_id=0,
    bos_id=1,
    eos_id=3,
    unk_id=4,
    control_symbols=["<sep>"],
    byte_fallback=True,
)


In [ ]:

sp = spm.SentencePieceProcessor(model_file="spm16k.model")

SPECIALS = ["<pad>", "<bos>", "<sep>", "<eos>", "<unk>"]
PAD = sp.pad_id()
BOS = sp.bos_id()
EOS = sp.eos_id()
UNK = sp.unk_id()
SEP = sp.piece_to_id("<sep>")
VOCAB_SIZE = sp.get_piece_size()

def encode_text(s: str):
    return sp.encode(s, out_type=int)

def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([i for i in ids if i not in drop])


In [ ]:

labeled = []
for text in corpus_texts:
    token_ids = [BOS] + encode_text(text) + [EOS]
    labeled.append((token_ids, token_ids.copy()))

print(f"Prepared sequences: {len(labeled)}")
if labeled:
    print("Sample token ids:", labeled[0][0][: min(len(labeled[0][0]), 32)])
    print("Sample decoded:", decode_text(labeled[0][0]))


Labeled pairs: 43870
Sample labeled pair: ([1, 378, 280, 309, 3901, 399, 744, 262, 2, 647, 263, 3], [-100, -100, -100, -100, -100, -100, -100, -100, -100, 647, 263, 3])


In [ ]:
def pack_token_sequences(sequences, max_length, pad_id):
    inputs, labels, attention_masks = [], [], []
    tokens_per_epoch = 0

    token_stream = []
    label_stream = []
    for tokens, lbls in sequences:
        token_stream.extend(tokens)
        label_stream.extend(lbls)

    for start in range(0, len(token_stream), max_length):
        end = min(start + max_length, len(token_stream))
        chunk_tokens = token_stream[start:end]
        chunk_labels = label_stream[start:end]
        attn = [1] * len(chunk_tokens)

        if len(chunk_tokens) < max_length:
            pad_len = max_length - len(chunk_tokens)
            chunk_tokens = chunk_tokens + [pad_id] * pad_len
            chunk_labels = chunk_labels + [-100] * pad_len
            attn = attn + [0] * pad_len

        tokens_per_epoch += sum(attn)
        inputs.append(chunk_tokens)
        labels.append(chunk_labels)
        attention_masks.append(attn)

    return inputs, labels, attention_masks, tokens_per_epoch


packed_inputs, packed_labels, packed_attn, tokens_per_epoch = pack_token_sequences(
    labeled, max_length=MAX_LEN, pad_id=PAD
)

print(f"Packed batches: {len(packed_inputs)}")
print(f"Tokens per epoch: {tokens_per_epoch}")
if packed_inputs:
    print("Sample packed input:", packed_inputs[0][: min(len(packed_inputs[0]), 32)])
    print("Sample packed labels:", packed_labels[0][: min(len(packed_labels[0]), 32)])
    print("Sample attention:", packed_attn[0][: min(len(packed_attn[0]), 32)])

Packed batches: 14449
Tokens per epoch: 2311734
Sample packed input: [1, 378, 280, 309, 3901, 399, 744, 262, 2, 647, 263, 3, 1, 273, 309, 342, 328, 267, 276, 767, 1662, 287, 13978, 328, 267, 276, 767, 1662, 287, 294, 362, 1028]
Sample packed labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, 647, 263, 3, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Sample attention: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


# 트렌스포머

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.nh = n_heads
        self.dk = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.nh, self.dk).transpose(1, 2)
        k = k.view(B, T, self.nh, self.dk).transpose(1, 2)
        v = v.view(B, T, self.nh, self.dk).transpose(1, 2)

        attn_bias = None
        if attn_mask is not None:
            if attn_mask.dim() != 2 or attn_mask.shape != (B, T):
                raise ValueError("attn_mask must be of shape (batch_size, sequence_length)")
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            neg_inf = torch.finfo(q.dtype).min
            attn_bias = pad_mask.to(dtype=q.dtype) * neg_inf

        dropout_p = self.attn_drop.p if self.training else 0.0
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_bias,
            dropout_p=dropout_p,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x


class GPTScratch(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=512,
        n_layers=8,
        n_heads=8,
        max_len=MAX_LEN,
        dropout=0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, 4, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init)
        self.head.weight = self.tok_emb.weight

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, attention_mask=None):
        B, T = x.shape
        if T > self.max_len:
            x = x[:, -self.max_len :]
            T = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len :]
        pos = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.drop(self.emb_norm(h))
        for blk in self.blocks:
            h = blk(h, attention_mask)
        h = self.ln_f(h)
        return self.head(h)

In [ ]:

def count_tokens_passages(texts, max_len=MAX_LEN):
    total = 0
    for text in texts:
        ids = [BOS] + encode_text(text) + [EOS]
        total += min(len(ids), max_len)
    return total

TOK_PER_EPOCH = count_tokens_passages(corpus_texts, max_len=MAX_LEN)

print(f"approx. tokens per epoch: {TOK_PER_EPOCH:,}")


≈ tokens per epoch: 2,311,734


# 학습

In [ ]:
import math
import os
import random
import time
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_CFG = {
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "betas": (0.9, 0.95),
    "accumulation_steps": 4,
    "max_grad_norm": 1.0,
    "warmup_steps": 500,
    "val_split": 0.1,
    "num_workers": 0,
    "pin_memory": bool(torch.cuda.is_available()),
    "seed": SEED,
    "checkpoint_dir": "molu_chatbot",
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(TRAIN_CFG["seed"])

class PackedDataset(Dataset):
    def __init__(self, inputs, labels, attention_masks):
        self.inputs = inputs
        self.labels = labels
        self.attention_masks = attention_masks

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.inputs[idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_masks[idx], dtype=torch.long),
        }

if not packed_inputs:
    raise RuntimeError("No packed sequences were produced; check preprocessing steps.")

full_dataset = PackedDataset(packed_inputs, packed_labels, packed_attn)
total_sequences = len(full_dataset)
if total_sequences < 2:
    raise RuntimeError(f"Need at least 2 packed sequences, but found {total_sequences}.")

val_size = max(1, int(total_sequences * TRAIN_CFG["val_split"]))
if val_size >= total_sequences:
    val_size = total_sequences - 1
train_size = total_sequences - val_size

generator = torch.Generator().manual_seed(TRAIN_CFG["seed"])
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

batch_size = TRAIN_CFG["batch_size"]
train_dl = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=TRAIN_CFG["num_workers"],
    pin_memory=TRAIN_CFG["pin_memory"],
)
val_dl = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=TRAIN_CFG["num_workers"],
    pin_memory=TRAIN_CFG["pin_memory"],
)

model = GPTScratch(
    vocab_size=VOCAB_SIZE,
    d_model=512,
    n_layers=8,
    n_heads=8,
    max_len=MAX_LEN,
    dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CFG["learning_rate"],
    betas=TRAIN_CFG["betas"],
    weight_decay=TRAIN_CFG["weight_decay"],
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def lm_loss(logits, targets):
    vocab = logits.size(-1)
    pred = logits[:, :-1].contiguous().view(-1, vocab)
    gold = targets[:, 1:].contiguous().view(-1)
    return F.cross_entropy(pred, gold, ignore_index=-100)

@torch.no_grad()
def evaluate(dataloader):
    model.eval()
    total_loss, batches = 0.0, 0
    for batch in dataloader:
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        attn = batch["attention_mask"].to(device)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            loss = lm_loss(model(x, attention_mask=attn), y)
        total_loss += loss.item()
        batches += 1
    return total_loss / max(1, batches)

accum_steps = TRAIN_CFG["accumulation_steps"]
updates_per_epoch = math.ceil(len(train_dl) / accum_steps)
total_updates = max(1, updates_per_epoch * TRAIN_CFG["epochs"])
warmup_steps = min(TRAIN_CFG["warmup_steps"], max(1, total_updates // 2))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = (step - warmup_steps) / float(max(1, total_updates - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val = float("inf")
history = []
global_step = 0

SKIP_TRAINING = bool(int(os.environ.get("SKIP_TRAINING", "0")))

if SKIP_TRAINING:
    print("SKIP_TRAINING=1 -> skipping training loop.")
else:
    for epoch in range(1, TRAIN_CFG["epochs"] + 1):
        model.train()
        start_time = time.time()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        tokens_seen = 0
        num_batches = len(train_dl)

        for batch_idx, batch in enumerate(train_dl, start=1):
            x = batch["input_ids"].to(device)
            y = batch["labels"].to(device)
            attn = batch["attention_mask"].to(device)
            tokens_seen += int(attn.sum().item())

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss = lm_loss(model(x, attention_mask=attn), y) / accum_steps

            scaler.scale(loss).backward()
            running_loss += loss.item() * accum_steps

            should_step = (batch_idx % accum_steps == 0) or (batch_idx == num_batches)
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

                if global_step % 50 == 0:
                    print(
                        f"step {global_step:05d} | "
                        f"loss {loss.item() * accum_steps:.4f} | "
                        f"lr {scheduler.get_last_lr()[0]:.2e}"
                    )

        avg_train_loss = running_loss / max(1, num_batches)
        train_ppl = math.exp(avg_train_loss) if avg_train_loss < 20 else float("inf")

        val_loss = evaluate(val_dl)
        val_ppl = math.exp(val_loss) if val_loss < 20 else float("inf")

        elapsed_minutes = (time.time() - start_time) / 60.0
        current_lr = scheduler.get_last_lr()[0]
        history.append(
            {
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "train_ppl": train_ppl,
                "val_loss": val_loss,
                "val_ppl": val_ppl,
                "lr": current_lr,
                "tokens": tokens_seen,
            }
        )

        print(
            f"[epoch {epoch:02d}] "
            f"train_loss={avg_train_loss:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_ppl={val_ppl:.2f} "
            f"tokens={tokens_seen:,} "
            f"lr={current_lr:.2e} "
            f"{elapsed_minutes:.1f} min"
        )

        if val_loss < best_val:
            best_val = val_loss
            checkpoint_dir = Path(TRAIN_CFG["checkpoint_dir"])
            checkpoint_dir.mkdir(parents=True, exist_ok=True)

            torch.save(model.state_dict(), checkpoint_dir / "model.pt")

            config_payload = {
                "vocab_size": VOCAB_SIZE,
                "d_model": 512,
                "n_layers": 8,
                "n_heads": 8,
                "max_len": MAX_LEN,
                "dropout": 0.1,
                "special_tokens": {"PAD": PAD, "BOS": BOS, "SEP": SEP, "EOS": EOS},
                "train_config": {**TRAIN_CFG, "betas": list(TRAIN_CFG["betas"])},
                "best_val_loss": best_val,
            }
            with open(checkpoint_dir / "config.json", "w", encoding="utf-8") as f_cfg:
                json.dump(config_payload, f_cfg, indent=2, ensure_ascii=False)
            with open(checkpoint_dir / "metrics.json", "w", encoding="utf-8") as f_metrics:
                json.dump(history, f_metrics, indent=2, ensure_ascii=False)

            print(f"saved checkpoint to {checkpoint_dir.resolve()}")

[epoch 1] val_loss=6.2280 | 0.6 min
SAVED: molu_chatbot
[epoch 2] val_loss=5.8190 | 0.6 min
SAVED: molu_chatbot
[epoch 3] val_loss=5.6568 | 0.6 min
SAVED: molu_chatbot
[epoch 4] val_loss=5.5327 | 0.6 min
SAVED: molu_chatbot
[epoch 5] val_loss=5.4093 | 0.6 min
SAVED: molu_chatbot
[epoch 6] val_loss=5.3106 | 0.6 min
SAVED: molu_chatbot
[epoch 7] val_loss=5.2399 | 0.6 min
SAVED: molu_chatbot
[epoch 8] val_loss=5.1867 | 0.6 min
SAVED: molu_chatbot
[epoch 9] val_loss=5.1560 | 0.6 min
SAVED: molu_chatbot
[epoch 10] val_loss=5.1382 | 0.6 min
SAVED: molu_chatbot
[epoch 11] val_loss=5.1367 | 0.6 min
SAVED: molu_chatbot
[epoch 12] val_loss=5.1594 | 0.6 min
[epoch 13] val_loss=5.1885 | 0.6 min
[epoch 14] val_loss=5.2269 | 0.6 min
[epoch 15] val_loss=5.2807 | 0.6 min
[epoch 16] val_loss=5.3376 | 0.6 min
[epoch 17] val_loss=5.4014 | 0.6 min
[epoch 18] val_loss=5.4706 | 0.6 min
[epoch 19] val_loss=5.5572 | 0.6 min
[epoch 20] val_loss=5.6261 | 0.6 min


# 인퍼런스

In [ ]:

import re
import json
import torch

SAVE_DIR = "molu_pretrain"  # 저장
device = "cuda" if torch.cuda.is_available() else "cpu"

# state_dict와 config 읽기
sd = torch.load(f"{SAVE_DIR}/model.pt", map_location="cpu")
with open(f"{SAVE_DIR}/config.json") as f:
    cfg = json.load(f)

# state_dict에서 실제 아키텍처 추론
def infer_arch_from_state_dict(state_dict):
    d_model = state_dict["tok_emb.weight"].shape[1]
    vocab_size = state_dict["tok_emb.weight"].shape[0]
    max_len = state_dict["pos_emb.weight"].shape[0]
    layer_idx = set()
    for k in state_dict.keys():
        m = re.match(r"blocks\.(\d+)\.", k)
        if m:
            layer_idx.add(int(m.group(1)))
    n_layers = (max(layer_idx) + 1) if layer_idx else cfg.get("n_layers", 6)
    return vocab_size, d_model, max_len, n_layers

vocab_size, d_model_sd, max_len_sd, n_layers_sd = infer_arch_from_state_dict(sd)

# n_heads 보정
def pick_heads(d_model, preferred=cfg.get("n_heads", 6)):
    divisors = [h for h in range(2, 65) if d_model % h == 0]
    if not divisors:
        raise ValueError(f"d_model={d_model}에 대한 유효한 n_heads가 없습니다.")
    if preferred in divisors:
        return preferred
    return min(divisors, key=lambda h: abs(h - preferred))

n_heads_fixed = pick_heads(d_model_sd, cfg.get("n_heads", 6))

print(f"[info] from state_dict: vocab={vocab_size}, d_model={d_model_sd}, max_len={max_len_sd}, n_layers={n_layers_sd}")
print(f"[info] cfg requested n_heads={cfg.get('n_heads', None)} -> using n_heads={n_heads_fixed}")

# GPTScratch
model = GPTScratch(
    vocab_size=vocab_size,
    d_model=d_model_sd,
    n_layers=n_layers_sd,
    n_heads=n_heads_fixed,
    max_len=max_len_sd,
    dropout=cfg.get("dropout", 0.1),
).to(device).eval()

# 가중치 로드
missing, unexpected = model.load_state_dict(sd, strict=False)
print("loaded. missing:", missing, "unexpected:", unexpected)


[info] from state_dict: vocab=16000, d_model=512, max_len=160, n_layers=8
[info] cfg requested n_heads=8 -> using n_heads=8
loaded. missing: [] unexpected: []


In [ ]:

# === text generation ===
import torch

HIST_MAX = MAX_LEN
MAX_NEW = 120

@torch.no_grad()
def sample_top_p(logits_row, top_p=0.9, temperature=0.8):
    logits_row = logits_row / max(1e-6, temperature)
    probs = torch.softmax(logits_row, dim=-1)
    sp, si = torch.sort(probs, descending=True)
    cum = torch.cumsum(sp, dim=-1)
    mask = cum > top_p
    mask[..., 0] = False
    sp = sp.masked_fill(mask, 0)
    if sp.sum() <= 0:
        return int(torch.argmax(probs).item())
    sp = sp / sp.sum()
    idx = torch.multinomial(sp, 1)
    return int(si[idx])

@torch.no_grad()
def generate_text(
    prompt: str,
    max_ctx: int = HIST_MAX,
    max_new_tokens: int = MAX_NEW,
    top_p: float = 0.9,
    temperature: float = 0.8,
):
    prompt_tokens = encode_text(prompt)
    history = [BOS] + prompt_tokens
    generated = []
    x = torch.tensor(history, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(x)
        nxt = sample_top_p(logits[0, -1], top_p=top_p, temperature=temperature)
        generated.append(nxt)
        history.append(nxt)
        if len(history) > max_ctx:
            history = history[-max_ctx:]
        x = torch.tensor(history, dtype=torch.long, device=device).unsqueeze(0)
        if nxt == EOS:
            break
    return decode_text(generated).strip()


# 채팅

In [ ]:

print(generate_text("오늘 아침 주요 신문 헤드라인:"))


요즘에 드라마를 많이 보는 거 같아. 근데 요즘 드라마를 보면 유튜브로 많이 보는 거 같아.
